# MindScope — Student Depression Dataset Analysis

Kaggle notebook for mkhoirulmustaqim/student-depression-detection-dataset. Attach it using Kaggle's Add Input control before running.

> Educational use only. This model is not a diagnosis or clinical screening tool. SHAP describes model behavior, not causes.

In [ ]:
import glob, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
RANDOM_STATE=42
TARGET_COLUMN=None  # Set explicitly if automatic detection does not find the target.
sns.set_theme(style='whitegrid',palette='viridis')
pd.set_option('display.max_columns',100)

## Load attached dataset
If multiple CSVs appear, select the correct one by changing DATA_PATH.

In [ ]:
csv_files=glob.glob('/kaggle/input/**/*.csv',recursive=True)
print('CSV files found:')
for i,p in enumerate(csv_files): print(i,p)
if not csv_files: raise FileNotFoundError('Use Kaggle Add Input to attach the Student Depression Detection Dataset.')
DATA_PATH=csv_files[0]
print('Loading:',DATA_PATH)
df=pd.read_csv(DATA_PATH)
print('Shape:',df.shape)
display(df.head())
display(pd.DataFrame({'dtype':df.dtypes.astype(str),'missing':df.isna().sum(),'unique':df.nunique(dropna=False)}))

## Dataset audit and EDA

In [ ]:
print('Rows, columns:',df.shape,' Exact duplicates:',df.duplicated().sum())
display(df.describe(include='all').T)
missing=df.isna().sum().sort_values(ascending=False); display(missing[missing>0].rename('missing_count').to_frame())
candidates=[c for c in df.columns if c.strip().lower() in {'depression','target','label','depressed'}]
print('Possible target columns:',candidates)
if TARGET_COLUMN is None:
    if not candidates: raise ValueError('Set TARGET_COLUMN to the label column name.')
    TARGET_COLUMN=candidates[0]
print('Target:',TARGET_COLUMN)
display(df[TARGET_COLUMN].value_counts(dropna=False).rename('count').to_frame())
ax=df[TARGET_COLUMN].value_counts(dropna=False).plot(kind='bar',figsize=(6,4),title='Target distribution'); ax.set_xlabel(TARGET_COLUMN); ax.set_ylabel('Records'); plt.tight_layout(); plt.show()
num_cols=df.select_dtypes(include=np.number).columns.drop(TARGET_COLUMN,errors='ignore').tolist()
cat_cols=[c for c in df.columns if c not in num_cols and c!=TARGET_COLUMN]
print('Numeric:',num_cols,'\nCategorical:',cat_cols)
if num_cols:
    display(df[num_cols].describe().T); df[num_cols].hist(figsize=(14,10),bins=25); plt.tight_layout(); plt.show()
for col in cat_cols:
    print('\n',col); display(df[col].value_counts(dropna=False).head(20).to_frame('count'))
if len(num_cols)>1:
    sns.heatmap(df[num_cols].corr(),cmap='vlag',center=0); plt.title('Numeric feature correlations'); plt.show()

## Target association and leakage review
Run this before modeling. Strong associations do not prove leakage, but near-perfect validation scores should prompt a review of how the label was defined and whether any input feature directly encodes it.

In [ ]:
# Audit one-feature target associations and category rules; this is diagnostic, not causal.
for col in [c for c in df.columns if c != TARGET_COLUMN]:
    print(f"\n{col}: target rate and count by value")
    tab = pd.crosstab(df[col].fillna('<MISSING>'), df[TARGET_COLUMN], normalize='index')
    counts = pd.crosstab(df[col].fillna('<MISSING>'), df[TARGET_COLUMN])
    display(pd.concat([counts.add_suffix('_count'), tab.add_suffix('_row_rate')], axis=1).round(3))
# Look for repeated predictor profiles with conflicting labels.
profile_cols = [c for c in df.columns if c != TARGET_COLUMN]
profile_counts = df.groupby(profile_cols, dropna=False)[TARGET_COLUMN].nunique()
print('Repeated predictor profiles with conflicting target labels:', int((profile_counts > 1).sum()))


> Associations are descriptive, not causal. Review dataset documentation and feature definitions before interpretation.

## Baseline model comparison
The audit found direct target leakage in `Health_condition`: all Critical records are labeled Yes and all Good records No. This field is excluded below. Baseline models include linear/tree methods and available XGBoost, CatBoost, and LightGBM packages. Models are compared on validation data; the test set is reserved for final evaluation.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, classification_report

data = df.drop_duplicates().copy()
# Exclude the feature that exactly reveals the target in this dataset.
EXCLUDED_FEATURES = ['Health_condition']
exclude_present = [c for c in EXCLUDED_FEATURES if c in data.columns and c != TARGET_COLUMN]
X = data.drop(columns=[TARGET_COLUMN] + exclude_present)
print('Excluded suspected leakage features:', exclude_present)
le = LabelEncoder()
y = le.fit_transform(data[TARGET_COLUMN].astype(str))
print('Label mapping:', dict(zip(le.classes_, le.transform(le.classes_))))
num = X.select_dtypes(include=np.number).columns.tolist()
cat = [c for c in X.columns if c not in num]
# Dense one-hot output keeps HistGradientBoosting compatible; this dataset is small enough for it.
pre = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat)
])
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=.20, random_state=RANDOM_STATE, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=.25, random_state=RANDOM_STATE, stratify=y_trainval)
print('Split sizes:', len(X_train), 'train;', len(X_val), 'validation;', len(X_test), 'test')
models = {
    'Dummy': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=250, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'HistGradientBoosting': HistGradientBoostingClassifier(random_state=RANDOM_STATE)
}
# Add popular boosting libraries when installed in the Kaggle runtime.
# If unavailable, the rest of the notebook still runs and reports the skipped model.
try:
    from xgboost import XGBClassifier
    models['XGBoost'] = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1,
        eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
    )
except ImportError:
    print('XGBoost unavailable; skipping it.')

try:
    from catboost import CatBoostClassifier
    models['CatBoost'] = CatBoostClassifier(
        iterations=400, depth=6, learning_rate=0.05,
        verbose=False, random_seed=RANDOM_STATE,
        allow_writing_files=False, thread_count=-1
    )
except ImportError:
    print('CatBoost unavailable; skipping it.')

try:
    from lightgbm import LGBMClassifier
    models['LightGBM'] = LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=15,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1
    )
except ImportError:
    print('LightGBM unavailable; skipping it.')
results = []
fitted = {}
for name, est in models.items():
    pipe = Pipeline([('prep', pre), ('model', est)])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    pred = pipe.predict(X_val)
    row = {'model': name, 'accuracy': accuracy_score(y_val, pred),
           'precision_macro': precision_score(y_val, pred, average='macro', zero_division=0),
           'recall_macro': recall_score(y_val, pred, average='macro', zero_division=0),
           'f1_macro': f1_score(y_val, pred, average='macro')}
    if hasattr(pipe, 'predict_proba') and len(np.unique(y_val)) == 2:
        prob = pipe.predict_proba(X_val)[:, 1]
        row['roc_auc'] = roc_auc_score(y_val, prob)
        row['pr_auc'] = average_precision_score(y_val, prob)
    results.append(row)
results_df = pd.DataFrame(results).sort_values('f1_macro', ascending=False)
display(results_df)
best_name = results_df.iloc[0]['model']
best = fitted[best_name]
print('Selected by validation macro F1:', best_name)


## Ensemble comparison: soft voting and stacking
Combine the available tree/boosting candidates using two standard ensemble strategies. Both are trained on training data and evaluated on validation data. Keep an ensemble only if it improves validation macro F1; test data remains untouched.

In [ ]:
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.model_selection import StratifiedKFold

ensemble_names = [name for name in [
    "HistGradientBoosting", "XGBoost", "CatBoost", "LightGBM", "Random Forest"
] if name in models]
ensemble_estimators = [
    (f"model_{i}", Pipeline([("prep", pre), ("model", models[name])]))
    for i, name in enumerate(ensemble_names)
]
print("Base models included:", ensemble_names)
ensemble_candidates = {}
if len(ensemble_estimators) >= 2:
    ensemble_candidates["Soft Voting"] = VotingClassifier(
        estimators=ensemble_estimators, voting="soft", n_jobs=-1
    )
    ensemble_candidates["Stacking"] = StackingClassifier(
        estimators=ensemble_estimators,
        final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced"),
        stack_method="predict_proba",
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=1
    )

ensemble_rows = []
for name, estimator in ensemble_candidates.items():
    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_val)
    prob = estimator.predict_proba(X_val)[:, 1]
    fitted[name] = estimator
    ensemble_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_val, pred),
        "precision_macro": precision_score(y_val, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_val, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_val, pred, average="macro"),
        "roc_auc": roc_auc_score(y_val, prob),
        "pr_auc": average_precision_score(y_val, prob)
    })
if ensemble_rows:
    results_df = pd.concat([results_df, pd.DataFrame(ensemble_rows)], ignore_index=True)
    display(results_df.sort_values("f1_macro", ascending=False))
    best_name = results_df.iloc[results_df["f1_macro"].argmax()]["model"]
    best = fitted[best_name]
    print("Selected after baseline and ensemble validation comparison:", best_name)
else:
    print("Not enough available base models to build ensembles; keeping the baseline comparison.")


## Tune the leading model (training data only)
This compact randomized search uses stratified cross-validation on the training partition. The validation set compares the tuned candidate with the baselines; the test set remains untouched until final evaluation. Tuning may or may not improve results.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

hgb_pipeline = Pipeline([
    ("prep", pre),
    ("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE))
])
param_distributions = {
    "model__learning_rate": [0.03, 0.05, 0.08, 0.1, 0.15],
    "model__max_iter": [100, 150, 200, 300],
    "model__max_leaf_nodes": [7, 15, 31, 63],
    "model__min_samples_leaf": [10, 20, 30, 50],
    "model__l2_regularization": [0.0, 0.1, 1.0, 5.0]
}
search = RandomizedSearchCV(
    estimator=hgb_pipeline,
    param_distributions=param_distributions,
    n_iter=16,
    scoring="f1_macro",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True
)
search.fit(X_train, y_train)
tuned_hgb = search.best_estimator_
print("Best training CV macro F1:", round(search.best_score_, 4))
print("Best parameters:", search.best_params_)

# Compare the tuned candidate on validation data; do not evaluate it on the test set here.
tuned_val_pred = tuned_hgb.predict(X_val)
tuned_val_row = {
    "model": "Tuned HistGradientBoosting",
    "accuracy": accuracy_score(y_val, tuned_val_pred),
    "precision_macro": precision_score(y_val, tuned_val_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_val, tuned_val_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_val, tuned_val_pred, average="macro")
}
tuned_val_prob = tuned_hgb.predict_proba(X_val)[:, 1]
tuned_val_row["roc_auc"] = roc_auc_score(y_val, tuned_val_prob)
tuned_val_row["pr_auc"] = average_precision_score(y_val, tuned_val_prob)
results_df = pd.concat([results_df, pd.DataFrame([tuned_val_row])], ignore_index=True)
display(results_df.sort_values("f1_macro", ascending=False))
if tuned_val_row["f1_macro"] > results_df.loc[results_df["model"] != "Tuned HistGradientBoosting", "f1_macro"].max():
    best_name = "Tuned HistGradientBoosting"
    best = tuned_hgb
    print("Tuned model selected on validation macro F1.")
else:
    print("Keeping the existing validation-selected baseline:", best_name)


## Validation threshold analysis
The default classifier threshold is 0.5. Compare thresholds on validation data only; this is an educational tradeoff analysis, not a clinical decision rule. The selected threshold maximizes validation macro F1 and must be fixed before final test evaluation.

In [ ]:
from sklearn.metrics import precision_recall_curve

if len(np.unique(y_val)) == 2 and hasattr(best, "predict_proba"):
    val_prob = best.predict_proba(X_val)[:, 1]
    threshold_rows = []
    for threshold in np.arange(0.10, 0.91, 0.01):
        val_pred = (val_prob >= threshold).astype(int)
        threshold_rows.append({
            "threshold": threshold,
            "macro_f1": f1_score(y_val, val_pred, average="macro"),
            "yes_precision": precision_score(y_val, val_pred, pos_label=1, zero_division=0),
            "yes_recall": recall_score(y_val, val_pred, pos_label=1, zero_division=0)
        })
    threshold_df = pd.DataFrame(threshold_rows)
    display(threshold_df.sort_values("macro_f1", ascending=False).head(10))
    selected_threshold = float(threshold_df.loc[threshold_df["macro_f1"].idxmax(), "threshold"])
    print("Threshold selected on validation data by macro F1:", round(selected_threshold, 2))
else:
    selected_threshold = 0.5
    print("Probability threshold unavailable; using estimator default.")


In [ ]:
# Final held-out test evaluation; selected model and threshold were chosen using validation data.
print("Selected model:", best_name)
print("Validation-selected threshold:", round(selected_threshold, 2))
if hasattr(best, "predict_proba") and len(np.unique(y_test)) == 2:
    test_prob = best.predict_proba(X_test)[:, 1]
    pred_default = (test_prob >= 0.5).astype(int)
    pred_selected = (test_prob >= selected_threshold).astype(int)
    print("\nTest results at default threshold 0.5:")
    print(classification_report(y_test, pred_default, target_names=le.classes_, zero_division=0))
    print("\nTest results at validation-selected threshold:")
    print(classification_report(y_test, pred_selected, target_names=le.classes_, zero_division=0))
    print("Test ROC-AUC:", round(roc_auc_score(y_test, test_prob), 4))
    print("Test PR-AUC:", round(average_precision_score(y_test, test_prob), 4))
else:
    pred_selected = best.predict(X_test)
    print(classification_report(y_test, pred_selected, target_names=le.classes_, zero_division=0))
sns.heatmap(confusion_matrix(y_test, pred_selected), annot=True, fmt="d", cmap="Greens")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title(f"{best_name} — test set at validation-selected threshold {selected_threshold:.2f}")
plt.show()


## Optional SHAP explanation
If needed, install SHAP with %pip install shap. Explanations are about model behavior, not medical causes.

In [ ]:
try:
    import shap
    background = best.named_steps['prep'].transform(X_train.sample(min(100, len(X_train)), random_state=RANDOM_STATE))
    sample = best.named_steps['prep'].transform(X_test.sample(min(200, len(X_test)), random_state=RANDOM_STATE))
    explainer = shap.Explainer(best.named_steps['model'], background)
    shap_values = explainer(sample)
    # Binary classifiers may return values for each class; explain the positive class when present.
    if len(shap_values.shape) == 3:
        shap.plots.beeswarm(shap_values[:, :, 1])
    else:
        shap.plots.beeswarm(shap_values)
except ImportError:
    print('SHAP is not installed; run %pip install shap, then rerun this cell.')
except Exception as e:
    print('SHAP could not run for this model/environment:', repr(e))


## Interpretation and limitations
Record findings after reviewing actual outputs and dataset documentation. Describe this as prediction of the dataset's recorded label. Discuss provenance, label definition, potential leakage, representativeness, and the limits of a random train/test split. Do not claim clinical validity or infer causation.